# Model judge experiment
Compares an LLM judge with the rule-based claim checker on the human-labelled claims.

Setup: Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom in one sitting (Colab clears files on reset).

The judge sees only the claim and building counts. No imagery is used. Do not commit the CSVs.

## 1. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

## 2. Download the models
Qwen3-8B is the same-family comparison. Gemma 3 12B and Phi-4 14B are independent families.

In [ ]:
MODELS = ['qwen3:8b', 'gemma3:12b', 'phi4:14b']
for m in MODELS:
    !ollama pull {m}

## 3. Upload `judge_input.csv`

In [ ]:
from google.colab import files
files.upload()  # choose judge_input.csv

import pandas as pd
df = pd.read_csv('judge_input.csv')
print(df.shape, df.in_49.sum(), 'claims in the 49-claim comparison set')
df.head()

## 4. Functions
Fixed prompt, temperature 0, fixed seed.

In [ ]:
import re, requests

PROMPT = """You check one claim about building damage in a satellite tile.
Evidence (counts of buildings with usable imagery):
total={n_total}, undamaged={n_none}, minor={n_minor}, major={n_major}, destroyed={n_destroyed}.
Quantity words: none = 0%, few = under 10%, some = 10-30%, many = 30-60%, most = over 60%.
'damaged' means minor + major + destroyed.
Claim: "{claim}"
Answer with JSON only: {{"label": "correct" | "incorrect" | "cannot_tell"}}"""

URL = "http://localhost:11434/api/generate"

def build_prompt(row):
    return PROMPT.format(**row.to_dict())

def call_judge(prompt, model):
    r = requests.post(URL, json={"model": model, "prompt": prompt, "stream": False,
                                 "think": False,
                                 "options": {"temperature": 0, "seed": 0}}, timeout=300)
    text = r.json().get("response", "")
    m = re.search(r'"label"\s*:\s*"(correct|incorrect|cannot_tell)"', text)
    return m.group(1) if m else "cannot_tell"

def run_judge(frame, model):
    out = frame.copy()
    out["judge"] = [call_judge(build_prompt(r), model) for _, r in frame.iterrows()]
    return out

def summarize(res):
    s = res[res.in_49]
    answered = s[s.judge != "cannot_tell"]
    agree = (answered.judge == answered.human_label).mean()
    false_conf = int(((s.judge == "correct") & (s.human_label == "incorrect")).sum())
    return {"n": len(s), "answered": len(answered), "agreement": round(float(agree), 3),
            "false_confirmations": false_conf, "judge_said_correct": int((s.judge == "correct").sum())}

def checker_baseline(frame):
    s = frame[frame.in_49]
    said_ok = s.checker_verdict == "supported"
    human_ok = s.human_label == "correct"
    return {"n": len(s), "agreement": round(float((said_ok == human_ok).mean()), 3),
            "false_confirmations": int((said_ok & ~human_ok).sum()),
            "checker_said_supported": int(said_ok.sum())}

def plot_summary(rows):
    import matplotlib.pyplot as plt
    t = pd.DataFrame(rows).T
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    t["agreement"].plot(kind="bar", ax=ax[0], title="Agreement with humans", ylim=(0, 1))
    t["false_confirmations"].plot(kind="bar", ax=ax[1], title="False confirmations")
    plt.tight_layout(); plt.show()

## 5. Smoke test on 3 rows
Every row should get a label. If all say `cannot_tell`, print `call_judge(build_prompt(df.iloc[0]), 'qwen3:8b')` to see the raw reply.

In [ ]:
print(run_judge(df.head(3), 'qwen3:8b')[['claim', 'human_label', 'judge']])

## 6. Run all models (about 2-3 minutes each)

In [ ]:
results = {}
for m in MODELS:
    results[m] = run_judge(df, m)
    results[m].to_csv(f"judge_{m.replace(':', '_')}.csv", index=False)
    print(m, summarize(results[m]))

## 7. Compare with the checker

In [ ]:
rows = {'checker (rules)': checker_baseline(df)}
for m in MODELS:
    rows[m] = summarize(results[m])
table = pd.DataFrame(rows).T
display(table)
plot_summary(rows)

## 8. Where the judges differ from humans
Disagreements for the best judge, to check whether they are wording problems or real errors.

In [ ]:
best = max(MODELS, key=lambda m: summarize(results[m])['agreement'])
r = results[best]
r = r[r.in_49 & (r.judge != r.human_label)]
print(best, len(r), 'disagreements')
r[['claim_id', 'claim', 'n_total', 'n_none', 'n_minor', 'n_major', 'n_destroyed', 'checker_verdict', 'human_label', 'judge']]

In [ ]:
def subset_summary(res, basis):
    s = res[res.in_49 & (res.basis == basis)]
    agree = int((s.judge == s.human_label).sum())
    ok = s.judge == "correct"
    false_conf = int((ok & (s.human_label == "incorrect")).sum())
    return {"n": len(s), "agree": agree, "agreement": round(agree / len(s), 3),
            "false_confirmations": false_conf, "judge_said_correct": int(ok.sum())}

def checker_subset_summary(frame, basis):
    s = frame[frame.in_49 & (frame.basis == basis)]
    said_ok = s.checker_verdict == "supported"
    human_ok = s.human_label == "correct"
    agree = int((said_ok == human_ok).sum())
    return {"n": len(s), "agree": agree, "agreement": round(agree / len(s), 3),
            "false_confirmations": int((said_ok & ~human_ok).sum()),
            "judge_said_correct": int(said_ok.sum())}

for basis in ["annotators agreed", "count rule single rater"]:
    rows = {"checker (rules)": checker_subset_summary(df, basis)}
    for m in MODELS:
        rows[m] = subset_summary(results[m], basis)
    print(basis)
    display(pd.DataFrame(rows).T)